### 对FI数据进行填充

In [ ]:
import pandas as pd
import numpy as np

fi = pd.read_excel('FI(季频).xlsx', engine='openpyxl')
stock_code_column = None
for col in 'code':
   if col in fi.columns:
       stock_code_column = col
       break
stock_code_column = fi.columns[0]
# 识别数值列
numeric_columns = fi.select_dtypes(include=[np.number]).columns.tolist()

def fill_missing_with_median_by_stock_improved(df, stock_col, numeric_cols):
   """
   按股票分组，用每支股票各指标的中位数填充缺失值
   如果该股票的某个指标完全没有数据，则用0填充
   """
   df_filled = df.copy()
  
   for col in numeric_cols:
       if col != stock_col:  # 避免对股票代码列进行填充
           # 第一步：按股票分组计算中位数并填充
           df_filled[col] = df_filled.groupby(stock_col)[col].transform(
               lambda x: x.fillna(x.median())
           )
           # 第二步：检查是否还有NaN值
           if df_filled[col].isnull().sum() > 0:
               df_filled[col] = df_filled[col].fillna(0)
  
   return df_filled

# 执行缺失值填充
fi_filled = fill_missing_with_median_by_stock_improved(fi, stock_code_column, numeric_columns)
fi_filled.to_csv('FI.csv', index=False, encoding='utf-8-sig')

### 对所有的股票数据进行整合，包括信息滞后等

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

class StockDataIntegrator:
    def __init__(self, fi_path, macro_path, trd_path_1, trd_path_2):
        """
        Args:
            fi_path: FI.csv 基本面数据
            macro_path: macroeconomic.xlsx 宏观经济数据
            trd_path_1: TRD_Dalyr(20150105-20200103).csv 
            trd_path_2: TRD_Dalyr(20200106-20241213).csv 
        """
        self.fi_path = fi_path
        self.macro_path = macro_path
        self.trd_path_1 = trd_path_1
        self.trd_path_2 = trd_path_2
        
        # 数据存储
        self.fi_data = None
        self.macro_data = None
        self.trading_data = None
        self.technical_data = None
        self.fundamental_data = None
        self.integrated_data = None
        
        # 基准股票-日期组合
        self.base_stock_dates = None
        self.fi_quarter_index = None
        self.macro_month_index = None
        
    def load_all_data(self):

        self.fi_data = pd.read_csv(self.fi_path)
        print(f"基本面数据形状: {self.fi_data.shape}")
        
        self.macro_data = pd.read_excel(self.macro_path)
        print(f"宏观数据形状: {self.macro_data.shape}")

        df1 = pd.read_csv(self.trd_path_1)
        df2 = pd.read_csv(self.trd_path_2)
        self.trading_data = pd.concat([df1, df2], ignore_index=True)
        
        # 数据预处理
        self.trading_data['Trddt'] = pd.to_datetime(self.trading_data['Trddt'])
        self.trading_data = self.trading_data.sort_values(['Stkcd', 'Trddt']).reset_index(drop=True)
        self.trading_data['Stkcd'] = self.trading_data['Stkcd'].apply(self._format_stock_code)
        
        print(f"交易数据形状: {self.trading_data.shape}")
        
        # 4. 创建基准股票-日期组合
        self._create_base_stock_dates()
        self._build_time_indexes()
        
        return True
    
    def _create_base_stock_dates(self):
        """创建基准的股票"""
        self.base_stock_dates = self.trading_data[['Stkcd', 'Trddt']].copy()
        self.base_stock_dates = self.base_stock_dates.rename(columns={
            'Stkcd': 'stock_code',
            'Trddt': 'date'
        })
        
        print(f"基准股票-日期组合: {len(self.base_stock_dates)} 行")
        print(f"唯一股票数量: {self.base_stock_dates['stock_code'].nunique()}")
        
        return self.base_stock_dates
    
    def _build_time_indexes(self):
        # 处理基本面数据
        fi_df = self.fi_data.copy()
        fi_df['EndDate'] = pd.to_datetime(fi_df['EndDate'] + '-01')
        fi_df['code'] = fi_df['code'].apply(self._format_stock_code)
        
        # 3月=Q1, 6月=Q2, 9月=Q3, 12月=Q4
        def month_to_quarter(date):
            year = date.year
            month = date.month
            if month == 3:
                return pd.Period(f'{year}Q1', 'Q')
            elif month == 6:
                return pd.Period(f'{year}Q2', 'Q')
            elif month == 9:
                return pd.Period(f'{year}Q3', 'Q')
            elif month == 12:
                return pd.Period(f'{year}Q4', 'Q')
            else:
                quarter_num = (month - 1) // 3 + 1
                return pd.Period(f'{year}Q{quarter_num}', 'Q')
        
        fi_df['quarter'] = fi_df['EndDate'].apply(month_to_quarter)

        self.fi_quarter_index = {}
        for _, row in fi_df.iterrows():
            stock_code = row['code']
            quarter = row['quarter']
            if stock_code not in self.fi_quarter_index:
                self.fi_quarter_index[stock_code] = {}
            if quarter not in self.fi_quarter_index[stock_code] or row['EndDate'] > self.fi_quarter_index[stock_code][quarter]['EndDate']:
                self.fi_quarter_index[stock_code][quarter] = row

        macro_df = self.macro_data.copy()
        macro_df['YearMonth'] = pd.to_datetime(macro_df['YearMonth'] + '-01')
        macro_df['year_month'] = macro_df['YearMonth'].dt.to_period('M')

        self.macro_month_index = {}
        for _, row in macro_df.iterrows():
            month = row['year_month']
            if month in self.macro_month_index:
                if row['YearMonth'] > self.macro_month_index[month]['YearMonth']:
                    self.macro_month_index[month] = row
            else:
                self.macro_month_index[month] = row
        
        print(f"基本面季度索引: {len(self.fi_quarter_index)} 只股票")
        print(f"宏观月度索引: {len(self.macro_month_index)} 个月份")
    
    def _format_stock_code(self, code):
        """格式化股票代码为6位数字格式"""
        code_int = int(float(code))
        return f"{code_int:06d}"

    def calculate_technical_indicators(self):
        """计算技术指标（严格按照基准股票-日期组合）"""
        print("\n=== 计算技术指标 ===")
        trading_df = self.trading_data.copy()
        trading_df = trading_df.rename(columns={
            'Stkcd': 'stock_code',
            'Trddt': 'date', 
            'Opnprc': 'open',
            'Hiprc': 'high',
            'Loprc': 'low', 
            'Clsprc': 'close',
            'Dnshrtrd': 'volume'
        })
        price_cols = ['open', 'high', 'low', 'close']
        for col in price_cols:
            trading_df = trading_df[trading_df[col] > 0]
        stock_groups = trading_df.groupby('stock_code')
        all_results = []

        for stock_code, group in stock_groups:
                
            group = group.sort_values('date').reset_index(drop=True)
            indicators = self._calculate_indicators_for_stock(group)
            all_results.append(indicators)
        self.technical_data = pd.concat(all_results, ignore_index=True)
        self.technical_data = self._fill_missing_values(self.technical_data)
        self.technical_data = pd.merge(
            self.base_stock_dates,
            self.technical_data,
            on=['stock_code', 'date'],
            how='left'
        )
        print(f"技术指标数据最终形状: {self.technical_data.shape}")
        
        return self.technical_data
    
    def _calculate_indicators_for_stock(self, df):
        """为单只股票计算技术指标"""
        result = df[['stock_code', 'date', 'open', 'high', 'low', 'close', 'volume']].copy()
        
        # 指标1-4: 收盘价对数收益率系列
        result['indicator_1'] = self._safe_log_ratio(df['close'], df['close'].shift(1))
        result['indicator_2'] = self._safe_log_ratio(df['close'].shift(1), df['close'].shift(2))
        result['indicator_3'] = self._safe_log_ratio(df['close'].shift(2), df['close'].shift(3))
        result['indicator_4'] = self._safe_log_ratio(df['close'].shift(3), df['close'].shift(4))
        
        # 指标5-8: 当期最高价与不同期开盘价的对数比率
        result['indicator_5'] = self._safe_log_ratio(df['high'], df['open'])
        result['indicator_6'] = self._safe_log_ratio(df['high'], df['open'].shift(1))
        result['indicator_7'] = self._safe_log_ratio(df['high'], df['open'].shift(2))
        result['indicator_8'] = self._safe_log_ratio(df['high'], df['open'].shift(3))
        
        # 指标9-11: 历史最高价与对应开盘价的对数比率
        result['indicator_9'] = self._safe_log_ratio(df['high'].shift(1), df['open'].shift(1))
        result['indicator_10'] = self._safe_log_ratio(df['high'].shift(2), df['open'].shift(2))
        result['indicator_11'] = self._safe_log_ratio(df['high'].shift(3), df['open'].shift(3))
        
        # 指标12-15: 最低价与开盘价的对数比率系列
        result['indicator_12'] = self._safe_log_ratio(df['low'], df['open'])
        result['indicator_13'] = self._safe_log_ratio(df['low'].shift(1), df['open'].shift(1))
        result['indicator_14'] = self._safe_log_ratio(df['low'].shift(2), df['open'].shift(2))
        result['indicator_15'] = self._safe_log_ratio(df['low'].shift(3), df['open'].shift(3))
        
        # 指标16: 动量指标
        result['indicator_16'] = self._calculate_momentum(df['close'], period=10)
        
        # 指标17: RSI
        result['indicator_17'] = self._calculate_rsi(df['close'], period=14)
        
        # 指标18: 真实波动幅度
        result['indicator_18'] = self._calculate_true_range(df['high'], df['low'], df['close'])
        
        # 指标19: ATR
        tr = self._calculate_true_range(df['high'], df['low'], df['close'])
        result['indicator_19'] = self._calculate_atr(tr, period=14)
        
        # 指标20: OBV
        result['indicator_20'] = self._calculate_obv(df['close'], df['volume'])
        
        return result
    
    def _safe_log_ratio(self, numerator, denominator):
        """安全的对数比率计算"""
        ratio = numerator / denominator
        ratio = ratio.where(ratio > 0, np.nan)
        return np.log(ratio)
    
    def _calculate_momentum(self, prices, period=10):
        """计算动量指标"""
        return (prices / prices.shift(period) - 1) * 100
    
    def _calculate_rsi(self, prices, period=14):
        """计算RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi
    
    def _calculate_true_range(self, high, low, close):
        """计算真实波动幅度"""
        prev_close = close.shift(1)
        tr1 = high - low
        tr2 = abs(high - prev_close)
        tr3 = abs(low - prev_close)
        true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        return true_range
    
    def _calculate_atr(self, true_range, period=14):
        """计算ATR"""
        return true_range.rolling(window=period).mean()
    
    def _calculate_obv(self, close, volume):
        """计算OBV"""
        price_change = close.diff()
        obv = pd.Series(index=close.index, dtype=float)
        obv.iloc[0] = volume.iloc[0]
        
        for i in range(1, len(close)):
            if price_change.iloc[i] > 0:
                obv.iloc[i] = obv.iloc[i-1] + volume.iloc[i]
            elif price_change.iloc[i] < 0:
                obv.iloc[i] = obv.iloc[i-1] - volume.iloc[i]
            else:
                obv.iloc[i] = obv.iloc[i-1]
        
        return obv
    
    def _fill_missing_values(self, df):
        """填充缺失值"""
        indicator_cols = [col for col in df.columns if col.startswith('indicator_')]
        
        def fill_group(group):
            group = group.sort_values('date')
            for col in indicator_cols:
                group[col] = group[col].ffill()
            return group
        
        filled_df = df.groupby('stock_code').apply(fill_group).reset_index(drop=True)
        return filled_df
    
    def _create_lagged_periods(self, quarter_period, month_period):
        """创建滞后时间周期的专用函数"""
        if isinstance(quarter_period, str):
            quarter_period = pd.Period(quarter_period, 'Q')
        if isinstance(month_period, str):
            month_period = pd.Period(month_period, 'M')
        
        # 计算滞后期：向前推一期
        lagged_quarter = quarter_period - 1
        lagged_month = month_period - 1
        
        return lagged_quarter, lagged_month
    
    def calculate_fundamental_indicators(self):
        """计算基本面指标"""
        print("\n=== 计算基本面指标===")
        
        fi_features = self.fi_data.columns[4:16]  # 第5列到第16列，共12列
        macro_features = self.macro_data.columns[1:6]  # 第2列到第6列，共5列 
        base_with_periods = self.base_stock_dates.copy()
        
        # 计算原始的时间周期
        base_with_periods['quarter'] = base_with_periods['date'].dt.to_period('Q')
        base_with_periods['year_month'] = base_with_periods['date'].dt.to_period('M')
        
        # 实现滞后效应
        lagged_quarters = []
        lagged_months = []
        
        for _, row in base_with_periods.iterrows():
            original_quarter = row['quarter']
            original_month = row['year_month']
            
            # 使用专用函数计算滞后期
            lagged_q, lagged_m = self._create_lagged_periods(original_quarter, original_month)
            lagged_quarters.append(lagged_q)
            lagged_months.append(lagged_m)
        
        base_with_periods['lagged_quarter'] = lagged_quarters
        base_with_periods['lagged_month'] = lagged_months
        
        # 批量处理函数
        def process_batch(batch_df):
            result_rows = []
            
            for _, row in batch_df.iterrows():
                stock_code = row['stock_code']
                trade_date = row['date']
                lagged_quarter = row['lagged_quarter']
                lagged_month = row['lagged_month']
                
                # 查找滞后的基本面数据
                fi_data = self._fast_find_quarter_data(stock_code, lagged_quarter)
                
                # 查找滞后的宏观数据
                macro_data = self._fast_find_month_data(lagged_month)

                # 计算真实数据
                row_data = self._create_fundamental_row(
                    stock_code, trade_date, fi_data, macro_data, fi_features, macro_features
                )
                
                result_rows.append(row_data)
            
            return result_rows

        batch_size = 10000
        all_results = []
        
        for i in range(0, len(base_with_periods), batch_size):
            batch_df = base_with_periods.iloc[i:i+batch_size]
            batch_results = process_batch(batch_df)
            all_results.extend(batch_results)
        self.fundamental_data = pd.DataFrame(all_results)
        print(f"总特征数: 12个原始 + 60个张量积 = 72个特征")  
        print(f"基本面数据形状: {self.fundamental_data.shape}")
      
        return self.fundamental_data
    
    def _fast_find_quarter_data(self, stock_code, target_quarter):
        """查找季度基本面数据"""
        if stock_code not in self.fi_quarter_index:
            return None
        
        stock_quarters = self.fi_quarter_index[stock_code]
        
        # 优先使用确切季度
        if target_quarter in stock_quarters:
            return stock_quarters[target_quarter]
        
        # 使用最近的历史季度
        available_quarters = [q for q in stock_quarters.keys() if q <= target_quarter]
        if available_quarters:
            latest_quarter = max(available_quarters)
            return stock_quarters[latest_quarter]

        return None
    
    def _fast_find_month_data(self, target_month):
        """查找月度宏观数据"""
        # 首先尝试找到确切月份
        if target_month in self.macro_month_index:
            return self.macro_month_index[target_month]
        
        # 如果没有确切月份的数据，使用向前填充（使用最近的历史数据）
        available_months = [m for m in self.macro_month_index.keys() if m <= target_month]
        if available_months:
            latest_month = max(available_months)
            return self.macro_month_index[latest_month]
        
        return None
    
    def _create_fundamental_row(self, stock_code, date, fi_data, macro_data, fi_features, macro_features):
        """创建基本面数据行"""
        row_data = {
            'stock_code': stock_code,
            'date': date
        }
        
        # 添加原始基本面指标（12列）
        for i, fi_feature in enumerate(fi_features, 1):
            row_data[f'fundamental_{i}'] = fi_data[fi_feature]
        
        # 计算张量积：基本面 ⊗ 宏观经济（12*5=60列）
        for i, fi_feature in enumerate(fi_features, 1):
            for j, macro_feature in enumerate(macro_features, 1):
                tensor_value = fi_data[fi_feature] * macro_data[macro_feature]
                row_data[f'tensor_{i}_{j}'] = tensor_value
        
        return row_data
    
    def integrate_all_data(self):

        tech_rows = len(self.technical_data)
        fund_rows = len(self.fundamental_data)
        print("\n=== 整合所有数据===")
        print(f"技术指标数据: {tech_rows} 行")
        print(f"基本面数据: {fund_rows} 行")

        # 准备技术指标列
        indicator_cols = [f'indicator_{i}' for i in range(1, 21)]
        existing_indicator_cols = [col for col in indicator_cols if col in self.technical_data.columns]
        
        # 选择技术指标数据的指标列
        tech_indicators = self.technical_data[['stock_code', 'date'] + existing_indicator_cols].copy()
        # 合并数据：由于行数已经一致且顺序相同，可以直接合并
        integrated = pd.merge(
            self.fundamental_data, 
            tech_indicators, 
            on=['stock_code', 'date'], 
            how='left'
        )
        
        self.integrated_data = integrated
        print(f"总特征数: 72个基本面 + {len(existing_indicator_cols)}个技术 = {72 + len(existing_indicator_cols)}个特征")
        print(f"整合数据形状: {self.integrated_data.shape}")
        
        return self.integrated_data
    
    def save_all_data(self, output_dir="./"):
        """保存所有数据"""
        print("\n=== 保存所有数据 ===")
        
        # 1. 保存基本面数据（72列）
        fundamental_path = f"{output_dir}fundamental_data_72_features_lagged.csv"
        self.fundamental_data.to_csv(fundamental_path, index=False, encoding='utf-8-sig')
        print(f"基本面数据已保存: {fundamental_path}")
        
        # 2. 保存技术面数据（只包含股票代码、日期和20个技术指标）
        # 选择技术指标列（不包含开盘价、收盘价等原始数据）
        indicator_cols = [f'indicator_{i}' for i in range(1, 21)]
        existing_indicator_cols = [col for col in indicator_cols if col in self.technical_data.columns]
        
        # 只保存股票代码、日期和技术指标列
        tech_columns = ['stock_code', 'date'] + existing_indicator_cols
        technical_data_clean = self.technical_data[tech_columns].copy()
        
        technical_path = f"{output_dir}technical_data_20_features.csv"
        technical_data_clean.to_csv(technical_path, index=False, encoding='utf-8-sig')
        print(f"技术指标数据已保存: {technical_path}")
        
        # 3. 保存整合数据（92列）
        integrated_path = f"{output_dir}integrated_data_92_features_lagged.csv"
        self.integrated_data.to_csv(integrated_path, index=False, encoding='utf-8-sig')
        print(f"整合数据已保存: {integrated_path}")
    
    def generate_summary_report(self):
        fund_rows = len(self.fundamental_data)
        print(f"\n基本面数据 (72个特征): {fund_rows} 行")
        print(f"日期范围: {self.fundamental_data['date'].min()} 到 {self.fundamental_data['date'].max()}")
        print(f"唯一股票数: {self.fundamental_data['stock_code'].nunique()}")

        tech_rows = len(self.technical_data)
        print(f"\n技术指标数据 (20个特征): {tech_rows} 行")
        print(f"日期范围: {self.technical_data['date'].min()} 到 {self.technical_data['date'].max()}")
        print(f"唯一股票数: {self.technical_data['stock_code'].nunique()}")

        int_rows = len(self.integrated_data)
        print(f"\n整合数据 (92个特征): {int_rows} 行")
        print(f"日期范围: {self.integrated_data['date'].min()} 到 {self.integrated_data['date'].max()}")
        print(f"唯一股票数: {self.integrated_data['stock_code'].nunique()}")
        
        print("\n" + "="*60)

def main():
    fi_path = "FI.csv"
    macro_path = "macroeconomic(月频).xlsx"
    trd_path_1 = "TRD_Dalyr(20150105-20200103).csv"
    trd_path_2 = "TRD_Dalyr(20200106-20241213).csv"
    integrator = StockDataIntegrator(fi_path, macro_path, trd_path_1, trd_path_2)

    # 1. 加载所有数据
    integrator.load_all_data()
    
    # 2. 计算技术指标
    integrator.calculate_technical_indicators()
    
    # 3. 计算基本面指标
    integrator.calculate_fundamental_indicators()
    
    # 4. 整合所有数据
    integrator.integrate_all_data()
    
    # 5. 保存所有数据
    integrator.save_all_data()
    
    # 6. 生成汇总报告
    integrator.generate_summary_report()
    
    return integrator
if __name__ == "__main__":
    result = main()

基本面数据形状: (6642, 16)
宏观数据形状: (121, 6)
交易数据形状: (377301, 8)
基准股票-日期组合: 377301 行
唯一股票数量: 162
基本面季度索引: 162 只股票
宏观月度索引: 121 个月份

=== 计算技术指标 ===
技术指标数据最终形状: (377301, 27)

=== 计算基本面指标===
总特征数: 12个原始 + 60个张量积 = 72个特征
基本面数据形状: (377301, 74)

=== 整合所有数据===
技术指标数据: 377301 行
基本面数据: 377301 行
总特征数: 72个基本面 + 20个技术 = 92个特征
整合数据形状: (377301, 94)

=== 保存所有数据 ===
基本面数据已保存: ./fundamental_data_72_features_lagged.csv
技术指标数据已保存: ./technical_data_20_features.csv
整合数据已保存: ./integrated_data_92_features_lagged.csv

基本面数据 (72个特征): 377301 行
日期范围: 2015-01-05 00:00:00 到 2024-12-13 00:00:00
唯一股票数: 162

技术指标数据 (20个特征): 377301 行
日期范围: 2015-01-05 00:00:00 到 2024-12-13 00:00:00
唯一股票数: 162

整合数据 (92个特征): 377301 行
日期范围: 2015-01-05 00:00:00 到 2024-12-13 00:00:00
唯一股票数: 162



### 删除上市时间晚于2014年的股票

In [2]:
import pandas as pd

def process_stock_data():
    df_fi = pd.read_excel("FI(季频).xlsx")

    df_filtered = df_fi[df_fi['EndDate'] == '2014-12'].copy()

    df_filtered['listingYear'] = pd.to_datetime(df_filtered['listingDate']).dt.year
    
    # 筛选出2014年大于上市年份的股票
    condition = df_filtered['listingYear'] > 2014
    stocks_to_remove = df_filtered[condition]['code'].tolist()
    
    print(f"需要删除的股票数量: {len(stocks_to_remove)}")
    print(f"删除后的股票数量: {len(df_filtered) - len(stocks_to_remove)}")
    print("需要删除的股票代码:")
    for i, stock in enumerate(stocks_to_remove):
        print(f"{i+1}. {int(stock):06d}")

    print("\n删除后的141个股票代码:")
    remaining_stocks = df_filtered[~condition]['code'].tolist()

    for i in range(0, len(remaining_stocks), 5):
        stock_batch = remaining_stocks[i:i+5]
        formatted_batch = [f"{int(s):06d}" for s in stock_batch]
        print(f"  {', '.join(formatted_batch)}")

    csv_files = [
        ("fundamental_data_72_features_lagged.csv", "FI汇总.csv"),
        ("technical_data_20_features.csv", "TI汇总.csv"),
        ("integrated_data_92_features_lagged.csv", "数据汇总.csv")
        
    ]
    
    for input_file, output_file in csv_files:
        df = pd.read_csv(input_file)
        print(f"原始数据形状: {df.shape}")

        original_count = len(df)
        df_cleaned = df[~df['stock_code'].isin(stocks_to_remove)]
        removed_count = original_count - len(df_cleaned)
        
        print(f"删除了 {removed_count} 行数据")
        print(f"剩余数据形状: {df_cleaned.shape}")

        df_cleaned.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"已保存为 {output_file}")


# 运行主函数
if __name__ == "__main__":
    process_stock_data()

需要删除的股票数量: 21
删除后的股票数量: 141
需要删除的股票代码:
1. 000166
2. 001979
3. 300433
4. 600025
5. 600919
6. 600926
7. 600958
8. 601021
9. 601066
10. 601138
11. 601211
12. 601229
13. 601360
14. 601838
15. 601878
16. 601881
17. 601985
18. 603259
19. 603799
20. 603833
21. 603986

删除后的141个股票代码:
  000001, 000002, 000063, 000100, 000157
  000333, 000338, 000425, 000538, 000568
  000625, 000651, 000661, 000725, 000768
  000776, 000786, 000858, 000876, 000895
  000938, 000963, 002001, 002007, 002027
  002050, 002142, 002179, 002230, 002236
  002241, 002252, 002271, 002304, 002311
  002352, 002415, 002460, 002475, 002493
  002555, 002594, 002601, 002714, 002736
  300015, 300033, 300059, 300122, 300124
  300142, 300408, 600000, 600009, 600010
  600011, 600015, 600016, 600018, 600019
  600028, 600029, 600030, 600031, 600036
  600048, 600050, 600061, 600085, 600104
  600111, 600115, 600176, 600196, 600276
  600309, 600332, 600346, 600362, 600406
  600436, 600438, 600519, 600547, 600570
  600585, 600588, 600660, 6